# 📊 Exploration du Dataset IMDb — Groupe G02
**Problématique P02 : Régularisation et Généralisation**  
**Modèle : BERT-base-uncased | Dataset : IMDb (D01)**

Ce notebook explore le dataset IMDb avant l'entraînement :
- Distribution des classes
- Distribution des longueurs
- Analyse de la tokenisation BERT
- Exemples représentatifs

## 0. Installation et imports

In [ ]:
# Installation des dépendances (Colab)
import os
if not os.path.exists('/content/PROJET G02 COMPLET'):
    raise FileNotFoundError('Clonez le dépôt dans /content/G02_PROJET avant de lancer ce notebook.')

!pip install -r '/content/PROJET G02 COMPLET'/requirements.txt -q
print('✅ Dépendances installées')

In [ ]:
import sys
sys.path.insert(0, '/content/PROJET G02 COMPLET/src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from data_loader import load_imdb_subset, explore_dataset, analyze_tokenization
from config import SEED, FIGURES_DIR

print('✅ Imports OK')

## 1. Chargement du dataset

In [ ]:
subsets = load_imdb_subset(
    num_train_per_class=300,
    num_val_per_class=100,
    num_test_per_class=150,
    seed=SEED,
    verbose=True
)
print(f"\nDataset chargé : {sum(len(v) for v in subsets.values())} exemples au total")

## 2. Distribution des classes

In [ ]:
import os

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Exploration Dataset IMDb (D01) — G02', fontweight='bold')

colors = ['#EF5350', '#42A5F5']

for idx, (split, data) in enumerate(subsets.items()):
    ax = axes[idx]
    labels  = [ex['label'] for ex in data]
    lengths = [len(ex['text'].split()) for ex in data]

    # Distribution des classes
    n_pos = sum(1 for l in labels if l == 1)
    n_neg = sum(1 for l in labels if l == 0)
    ax.bar(['Négatif', 'Positif'], [n_neg, n_pos], color=colors, edgecolor='black')
    ax.set_title(f'{split.capitalize()} ({len(data)} ex.)')
    ax.set_ylabel('Nombre d\'exemples')
    ax.grid(True, axis='y', alpha=0.3)
    for bar, val in zip(ax.patches, [n_neg, n_pos]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

# Create the 'figures' directory if it doesn't exist
output_dir = '../figures'
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'exploration_classes.png'), dpi=150, bbox_inches='tight')
plt.show()

# Distribution des longueurs
fig, ax = plt.subplots(figsize=(10, 4))
for split, data in subsets.items():
    lengths = [len(ex['text'].split()) for ex in data]
    ax.hist(lengths, bins=40, alpha=0.6, label=split)
ax.axvline(200, color='red', linestyle='--', linewidth=2, label='max_length=200')
ax.set_xlabel('Longueur (mots)'); ax.set_ylabel('Fréquence')
ax.set_title('Distribution des longueurs — IMDb')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'exploration_longueurs.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Troncature à 200 tokens : conserve environ',
      f"{sum(1 for ex in subsets['train'] if len(ex['text'].split()) <= 200) / len(subsets['train']):.0%}",
      'des exemples d\'entraînement intacts.')

## 3. Distribution des longueurs

In [ ]:
from data_loader import explore_dataset
stats = explore_dataset(subsets, max_length=200, save=True)
plt.show()

## 4. Analyse de la tokenisation BERT

In [ ]:
from data_loader import analyze_tokenization
analyze_tokenization(subsets, n_samples=10)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('bert-base-uncased')

FIGURES_DIR = '../figures' # Define FIGURES_DIR

ratios = []
for ex in subsets['train']:
    words  = len(ex['text'].split())
    tokens = len(tok.encode(ex['text'], add_special_tokens=False))
    ratios.append(tokens / max(words, 1))

print(f'Ratio mots→tokens moyen : {np.mean(ratios):.3f}')
print(f'Max_length=200 tokens ≈ {200/np.mean(ratios):.0f} mots réels')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ratios, bins=30, color='#3498DB', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(ratios), color='red', linestyle='--', label=f'Moyenne = {np.mean(ratios):.3f}')
ax.set_xlabel('Ratio tokens/mots')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution du ratio de tokenisation BERT (WordPiece)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/explore_tokenization_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Exemples représentatifs

In [ ]:
print('=== 3 critiques POSITIVES ===\n')
pos_examples = [ex for ex in subsets['train'] if ex['label'] == 1][:3]
for i, ex in enumerate(pos_examples, 1):
    print(f'[{i}] {ex["text"][:200]}...\n')

print('\n=== 3 critiques NÉGATIVES ===\n')
neg_examples = [ex for ex in subsets['train'] if ex['label'] == 0][:3]
for i, ex in enumerate(neg_examples, 1):
    print(f'[{i}] {ex["text"][:200]}...\n')

## 6. Synthèse

| Caractéristique | Valeur |
|---|---|
| Classes | 2 (pos/nég) — parfaitement équilibrées |
| Train / Val / Test | 600 / 200 / 300 |
| Longueur moy. (mots) | ~235 mots |
| Ratio tokenisation | ~1.24 tokens/mot |
| max_length=200 tokens | ≈ 161 mots réels |
| Exemples tronqués | ~42% du train |